In [1]:
import pandas as pd
import numpy as np
import os

In [3]:
thermophile_dataset = pd.read_pickle('thermophile2.pkl')
boundaries = thermophile_dataset['boundary'].str.split(":", expand=True)
thermophile_dataset['discontinuous'] = boundaries[2].notnull()
thermophile_dataset = thermophile_dataset.drop('boundary', axis=1)
thermophile_dataset

,fold,domain,taxid,PDB,therm,taxonomy,tax_domain,discontinuous
0,a.1.1.1,d1ux8a_,1423,1UX8_A,False,Bacteria,Bacteria,False
1,a.1.1.1,d1dlwa_,1423,1DLW_A,False,Bacteria,Bacteria,False
3,a.1.1.1,d1dlya_,1423,1DLY_A,False,Bacteria,Bacteria,False
4,a.1.1.1,d1uvxa_,1423,1UVX_A,False,Bacteria,Bacteria,False
18,a.1.1.1,d1s56b_,1773,1S56_B,False,Bacteria,Bacteria,False
...,...,...,...,...,...,...,...,...
302293,g.80.1.1,d1wfha1,727,1WFH_A,False,Bacteria,Bacteria,False
302294,g.80.1.1,d1wfla1,727,1WFL_A,False,Bacteria,Bacteria,False
302295,g.81.1.1,d1vzya2,1423,1VZY_A,False,Bacteria,Bacteria,False
302296,g.81.1.1,d1vzyb2,1423,1VZY_B,False,Bacteria,Bacteria,False


Make dictionary of the SCOP domains (have to subset the PDB chain with just the residues in the domain)

In [5]:
AA_dict = {i:j for i,j in zip(['a','r','n','d','c','q','e','g','h','i','l','k','m','f','p','s','t','w','y','v','x'],range(21))}
AA_dict['X'] = -9

astral_file = open("./data/astral-scopedom-seqres-gd-all-2.08-stable.fa", "r")
domain_dict = {}

line = next(astral_file)
dom_name = line.split()[0][1:]
seq = ""
domain_list = list(thermophile_dataset['domain'])

for line in astral_file:
    if line[0] == ">":
        if dom_name in domain_list:
            try:
                domain_dict[dom_name] = np.array([AA_dict[x] for x in seq], dtype=np.int8)
                seq = ""
                dom_name = line.split()[0][1:]
            except Exception as e:
                print(dom_name, e, seq)
                seq = ""
                dom_name = line.split()[0][1:]
        else:
            seq = ""
            dom_name = line.split()[0][1:]
    else:
        seq = seq + line.strip()

astral_file.close()
# np.savez("domain_dict.npz", **domain_dict)
# domain_dict = np.load("domain_dict.npz")

d2fmda_ 'b' ansvcftftdfesgqqdlifqgdasvgsnkalqltkvdskgnpqggsvgralytapirlwqssslvasfettftfsisqgsstpadaltffiaspdtkipsgsggrllglfgssnnagsdngvvsvefdtypntdigdpnyrhigidvnsirskaaskwdwqngktatahisynsaskrlsvvssypnsspvvvsfdvelnnvbpbwvrvgfsattgqytqtnnilawsfrsslmg
d1kp0a1 'z' aamiztzkyhngzkkytpfszaemtrrzbrlrawmaksbidavlftsyhninyysgwlycyfgrkyazvibzvkavtiskgidggmpwrrsfgbnivytdwkrdnfysavkklvkgakzigiehdhvtlbhrrzlzkalpgtefvdvgzpvmwzrv
d1kp0a2 'z' iksseezblirzgarisdiggaataaaisagvpeyevaiattbamvrziarbfpyvelmdtwiwfqsgintdgahnpvtbrvvzrgdilslncfpmifgyytalertlflzzvbdaslziwzkntavhrrglzlikpgarckdiaselnbmyrzwdllryrtfgyghsfgvlbhyygreagvelrediztvlepgmvvsmepmvmbpegepgaggyrehdilvikenbtenitgfpfgpehniika


In [8]:
thermophile_dataset.drop(thermophile_dataset[thermophile_dataset['domain'].isin(['d2fmda_','d1kp0a1','d1kp0a2'])].index , inplace=True)

In [10]:
MLP_predictions = np.load("../predictions/MLP_predictions.npz")
HMM_predictions = np.load("../predictions/HMM_predictions.npz") 
PLM_predictions = np.load("../predictions/PLM_predictions.npz")
PSSM_predictions= np.load("../predictions/PSSM_predictions.npz")
all_chains      = np.load("../predictions/all_chains.npz")

In [11]:
import numba as nb

@nb.jit
def _is_equal_nb(seq, subseq, m, i):
    for j in range(m):
        if seq[i + j] != subseq[j]:
            return False
    return True

@nb.jit
def find_loop_nb(seq, subseq):
    n = len(seq)
    m = len(subseq)
    for i in range(n - m + 1):
        if _is_equal_nb(seq, subseq, m, i):
            yield i

In [12]:
def get_subchain_accuracy(stride, domain_len, prediction, ground_truth):
    prediction = prediction[stride:stride+domain_len]
    ground_truth = ground_truth[stride:stride+domain_len]
    mask = ground_truth!=-9
    prediction = prediction[mask]
    ground_truth = ground_truth[mask]
    assert len(prediction)==len(ground_truth)
    return (sum(prediction==ground_truth)/len(prediction))

In [13]:
thermophile_dataset = thermophile_dataset[~thermophile_dataset['discontinuous']]

In [14]:
thermophile_dataset['MLP'] = None
thermophile_dataset['PLM'] = None
thermophile_dataset['PSSM']= None
thermophile_dataset['HMM'] = None

for row in thermophile_dataset.index:
    PDB_ID = thermophile_dataset.loc[row, 'PDB']
    dom_ID = thermophile_dataset.loc[row, 'domain']
    
    if dom_ID in domain_dict.keys():
        if PDB_ID in all_chains.files:
            try:
                chain_seq = all_chains[PDB_ID][0]
                domain_seq= domain_dict[dom_ID]
                stride = next(find_loop_nb(chain_seq, domain_seq))

                domain_len = len(domain_seq)
                ground_truth = all_chains[PDB_ID][1]
                thermophile_dataset.at[row, 'MLP'] = get_subchain_accuracy(stride, domain_len, MLP_predictions[PDB_ID], ground_truth)
                thermophile_dataset.at[row, 'HMM'] = get_subchain_accuracy(stride, domain_len, HMM_predictions[PDB_ID], ground_truth)
                thermophile_dataset.at[row, 'PLM'] = get_subchain_accuracy(stride, domain_len, PLM_predictions[PDB_ID], ground_truth)
                thermophile_dataset.at[row, 'PSSM']= get_subchain_accuracy(stride, domain_len, PSSM_predictions[PDB_ID], ground_truth)
            except Exception as e:
                pass
#                 print(PDB_ID, e)


C:\Users\Mohan2\AppData\Local\Temp\ipykernel_16768\371561831.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  thermophile_dataset['MLP'] = None
C:\Users\Mohan2\AppData\Local\Temp\ipykernel_16768\371561831.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  thermophile_dataset['PLM'] = None
C:\Users\Mohan2\AppData\Local\Temp\ipykernel_16768\371561831.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

Se

In [19]:
thermophile_dataset = thermophile_dataset[thermophile_dataset['PSSM'].notnull()]

In [20]:
thermophile_dataset.to_csv("../results/data/thermophile_prokaryote.csv")

In [43]:
def get_discontinuous_accuracy(chain, subchain, prediction, ground_truth):
    split = np.where(subchain==-9)
    split = split[0]
    subchains = np.split(subchain, split)
    subchains[1] = subchains[1][1:]
    stride = next(find_loop_nb(chain, subchains[0]))
    acc1 = get_subchain_accuracy(stride, len(subchains[0]), prediction, ground_truth) * len(subchains[0])
    stride = next(find_loop_nb(chain, subchains[1]))
    acc2 = get_subchain_accuracy(stride, len(subchains[1]), prediction, ground_truth) * len(subchains[1])
    return (acc1+acc2)/(len(subchains[0])+len(subchains[1]))

In [45]:
for row in thermophile_dataset[thermophile_dataset['discontinuous']].index:
    PDB_ID = thermophile_dataset.loc[row, 'PDB']
    dom_ID = thermophile_dataset.loc[row, 'domain']
    
    if dom_ID in domain_dict.keys():
        if PDB_ID in all_chains.files:
            try:
                chain_seq = all_chains[PDB_ID][0]
                domain_seq= domain_dict[dom_ID]

                domain_len = len(domain_seq)
                ground_truth = all_chains[PDB_ID][1]
                thermophile_dataset.at[row, 'MLP'] = get_discontinuous_accuracy(chain_seq, domain_seq, MLP_predictions[PDB_ID], ground_truth)
                thermophile_dataset.at[row, 'HMM'] = get_discontinuous_accuracy(chain_seq, domain_seq, HMM_predictions[PDB_ID], ground_truth)
                thermophile_dataset.at[row, 'PLM'] = get_discontinuous_accuracy(chain_seq, domain_seq, PLM_predictions[PDB_ID], ground_truth)
                thermophile_dataset.at[row, 'PSSM']= get_discontinuous_accuracy(chain_seq, domain_seq, PSSM_predictions[PDB_ID], ground_truth)
            except Exception as e:
                print(PDB_ID, e)

1MJE_A '1MJE_A is not a file in the archive'
1K32_C '1K32_C is not a file in the archive'


In [46]:
thermophile_dataset[thermophile_dataset['discontinuous']]

,fold,domain,taxid,PDB,therm,tax_group,discontinuous,MLP,PLM,PSSM,HMM
39052,a.129.1.1,d1kp8e1,562,1KP8_E,False,Enterobacterales Enterobacteriaceae Escherichia,True,0.599206,0.857143,0.797619,0.59127
39188,a.129.1.1,d1sjpa1,1773,1SJP_A,False,Mycobacterium tuberculosis complex,True,0.697096,0.829236,0.793694,0.642836
39189,a.129.1.1,d1sjpb1,1773,1SJP_B,False,Mycobacterium tuberculosis complex,True,0.651601,0.838557,0.780584,0.633514
39195,a.129.1.1,d1iokf1,266,1IOK_F,False,Rhodobacterales Rhodobacteraceae Paracoccus,True,0.619048,0.916667,0.793651,0.56746
81153,b.29.1.2,d1axka1,44252,1AXK_A,False,Bacillales Paenibacillaceae Paenibacillus,True,0.701923,0.831731,0.730769,0.451923
...,...,...,...,...,...,...,...,...,...,...,...
248045,d.127.1.1,d1xgsb2,2261,1XGS_B,True,Thermococci Thermococcaceae Pyrococcus,True,0.711009,0.899083,0.802752,0.655963
248051,d.127.1.1,d1xgmb2,2261,1XGM_B,True,Thermococci Thermococcaceae Pyrococcus,True,0.692661,0.880734,0.802752,0.642202
248329,d.128.1.5,d2d6fd3,145262,2D6F_D,True,Methanobacteria Methanobacteriaceae Methanothe...,True,None,None,None,None
248331,d.128.1.5,d1zq1d3,29292,1ZQ1_D,True,Thermococci Thermococcaceae Pyrococcus,True,0.629283,0.813084,0.766355,0.566978


In [48]:
thermophile_dataset = thermophile_dataset[thermophile_dataset['PSSM'].notnull()]
thermophile_dataset = thermophile_dataset[thermophile_dataset['fold'].isin(thermophile_dataset[thermophile_dataset['therm']]['fold'].unique())]
thermophile_dataset

,fold,domain,taxid,PDB,therm,tax_group,discontinuous,MLP,PLM,PSSM,HMM
0,a.1.1.1,d1ux8a_,1423,1UX8_A,False,Bacillus subtilis group,False,0.788136,0.898305,0.813559,0.779661
1,a.1.1.1,d1dlwa_,5885,1DLW_A,False,Peniculida Parameciidae Paramecium,False,0.706897,0.931034,0.818966,0.603448
3,a.1.1.1,d1dlya_,3054,1DLY_A,False,Chlamydomonadales Chlamydomonadaceae Chlamydom...,False,0.785124,0.917355,0.752066,0.719008
4,a.1.1.1,d1uvxa_,3054,1UVX_A,False,Chlamydomonadales Chlamydomonadaceae Chlamydom...,False,0.752066,0.92562,0.760331,0.710744
18,a.1.1.1,d1s56b_,1773,1S56_B,False,Mycobacterium tuberculosis complex,False,0.674074,0.925926,0.785185,0.511111
...,...,...,...,...,...,...,...,...,...,...,...
302095,g.59.1.1,d1k81a_,2190,1K81_A,False,Methanococcales Methanocaldococcaceae Methanoc...,False,0.333333,0.888889,0.638889,0.75
302295,g.81.1.1,d1vzya2,1423,1VZY_A,False,Bacillus subtilis group,False,0.684211,0.894737,0.807018,0.596491
302296,g.81.1.1,d1vzyb2,1423,1VZY_B,False,Bacillus subtilis group,False,0.698113,0.811321,0.773585,0.471698
302297,g.81.1.1,d1xjha_,562,1XJH_A,False,Enterobacterales Enterobacteriaceae Escherichia,False,0.774194,0.870968,0.741935,0.612903


In [49]:
thermophile_dataset.to_csv("thermophile.csv")